# SMX Single-Surface Hybrid-DFT + IEF-PCM(water) Frequencies

Thin disposable launcher for `research/smx-single-surface-thermochemistry/run_colab_dft.py`.

**Recommended Colab Runtime:** `Runtime -> Change runtime type -> A100 GPU` (native `FP64` Tensor Cores; ~39x faster double-precision throughput than T4).

In [ ]:
# Cell 1: Verify GPU (aim for A100), check CUDA version, and mount Google Drive for checkpointing
!nvidia-smi
!nvcc --version
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/pvjohnston-colab/smx-single-surface-thermochemistry'

In [ ]:
# Cell 2: Auto-detect CUDA 12 vs 13, remove conflicting CuPy wheels, and enable cuTENSOR
import glob, os, re, site, subprocess, sys

nvcc_out = subprocess.check_output(['nvcc', '--version'], text=True)
m = re.search(r'release (\d+)\.', nvcc_out)
cuda_major = int(m.group(1)) if m else 12
print(f'Detected CUDA major version: {cuda_major}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
if cuda_major >= 13:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'cupy-cuda12x', 'cutensor-cu12', 'gpu4pyscf-cuda12x', 'gpu4pyscf-libxc-cuda12x'], check=False)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyscf', 'gpu4pyscf-cuda13x', 'cutensor-cu13', 'cupy-cuda13x', 'pyscf-dispersion', 'geometric'], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'cupy-cuda13x', 'cutensor-cu13', 'gpu4pyscf-cuda13x', 'gpu4pyscf-libxc-cuda13x'], check=False)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyscf', 'gpu4pyscf-cuda12x', 'cutensor-cu12', 'cupy-cuda12x', 'pyscf-dispersion', 'geometric'], check=True)

# Expose cuTENSOR and NVIDIA shared libraries to all subprocesses
lib_dirs = []
for sp in site.getsitepackages():
    lib_dirs.extend(sorted(glob.glob(os.path.join(sp, 'cutensor', 'lib'))))
    lib_dirs.extend(sorted(glob.glob(os.path.join(sp, 'nvidia', '*', 'lib'))))
if lib_dirs:
    os.environ['LD_LIBRARY_PATH'] = os.pathsep.join(lib_dirs) + os.pathsep + os.environ.get('LD_LIBRARY_PATH', '')

subprocess.run([sys.executable, '-c', (
    'import cupy as cp; '
    'props = cp.cuda.runtime.getDeviceProperties(0); '
    'import gpu4pyscf.lib.cutensor as ct; '
    'print("GPU:", props["name"].decode(), "| SM:", f"{props[\x27major\x27]}.{props[\x27minor\x27]}", "| contract_engine:", getattr(ct, "contract_engine", "loaded"))'
)], check=True)

In [ ]:
# Cell 3: Clone the frozen branch from GitHub and run the H2O smoke test (<15 s on A100)
%cd /content
!rm -rf pvjohnston.com
!git clone --depth 1 -b post/smx-single-surface-thermochemistry https://github.com/pvjohnston/pvjohnston.com.git
%cd /content/pvjohnston.com
!python3 research/smx-single-surface-thermochemistry/thermo_analysis.py --preflight
!python3 research/smx-single-surface-thermochemistry/run_colab_dft.py --smoke-test --checkpoint-dir "$CHECKPOINT_DIR"

In [ ]:
# Cell 4: Run SMX conformers A, B, C, D (checkpointed after each conformer) and download artifact bundle
%cd /content/pvjohnston.com
!python3 research/smx-single-surface-thermochemistry/run_colab_dft.py --checkpoint-dir "$CHECKPOINT_DIR"

from google.colab import files
files.download('/content/pvjohnston.com/research/smx-single-surface-thermochemistry/smx_single_surface_colab_artifacts.tar.gz')